# Malaika — Fine-Tune Gemma 4 for Breath Sound Classification (v2 Stronger)

**Pipeline**: ICBHI Audio → mel-spectrogram PNG → Gemma 4 vision LoRA fine-tuning

**v2 changes from v1**: Higher LoRA rank (r=32), more target modules (q/k/v/o_proj), 300 steps, dropout 0.05

| | v1 (failed) | v2 (this run) |
|--|--|--|
| LoRA rank | r=8 | r=32 |
| Target modules | q_proj, v_proj | q_proj, k_proj, v_proj, o_proj |
| Training steps | 100 | 300 |
| Dropout | 0.0 | 0.05 |
| v1 accuracy | 20% (worse than 25% baseline) | TBD |

**Colab secrets needed**: `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY` | **GPU**: T4

In [ ]:
# Cell 1: Install
!pip install -q git+https://github.com/huggingface/transformers.git peft trl datasets bitsandbytes accelerate librosa Pillow soundfile kaggle

In [ ]:
# Cell 2: Auth + imports
from huggingface_hub import login
import os, subprocess, json, random, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import librosa
from PIL import Image

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    login(token=secrets.get_secret("HF_TOKEN"))
    KU, KK = secrets.get_secret("KAGGLE_USERNAME"), secrets.get_secret("KAGGLE_KEY")
    print("Auth: Kaggle")
except ModuleNotFoundError:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
        KU, KK = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        print("Auth: Colab")
    except Exception:
        login()
        KU, KK = os.environ.get("KAGGLE_USERNAME", ""), os.environ.get("KAGGLE_KEY", "")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KU, "key": KK}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")

In [ ]:
# Cell 3: Download ICBHI
ICBHI_DIR = Path("/tmp/icbhi_data")
ICBHI_INNER = ICBHI_DIR / "Respiratory_Sound_Database" / "Respiratory_Sound_Database"
KAGGLE_NATIVE = Path("/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database")

if KAGGLE_NATIVE.exists():
    audio_dir = KAGGLE_NATIVE / "audio_and_txt_files"
elif ICBHI_INNER.exists():
    audio_dir = ICBHI_INNER / "audio_and_txt_files"
else:
    print("Downloading ICBHI...")
    ICBHI_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "vbookshelf/respiratory-sound-database",
        "-p", str(ICBHI_DIR), "--unzip"], capture_output=True, text=True, check=True)
    audio_dir = ICBHI_INNER / "audio_and_txt_files"

audio_files = list(audio_dir.glob("*.wav"))
print(f"ICBHI: {len(audio_files)} audio files")

In [ ]:
# Cell 4: Parse annotations
def parse_icbhi_annotations(adir):
    records = []
    for txt in sorted(adir.glob("*.txt")):
        wav = txt.with_suffix(".wav")
        if not wav.exists(): continue
        pid = txt.stem.split("_")[0]
        hc, hw = False, False
        with open(txt) as f:
            for line in f:
                ff = line.strip().split()
                if len(ff) >= 4:
                    if int(ff[2]) == 1: hc = True
                    if int(ff[3]) == 1: hw = True
        label = "both" if hc and hw else "crackle" if hc else "wheeze" if hw else "normal"
        records.append({"audio_path": str(wav), "patient_id": pid,
            "label": label, "has_crackle": hc, "has_wheeze": hw})
    return records

records = parse_icbhi_annotations(audio_dir)
print(f"Parsed {len(records)} recordings:")
for label, count in sorted(Counter(r["label"] for r in records).items()):
    print(f"  {label}: {count}")

In [ ]:
# Cell 5: Generate spectrograms
SPEC_DIR = Path("/tmp/spectrograms")
SPEC_DIR.mkdir(exist_ok=True)
SPEC_W, SPEC_H = 512, 256

def audio_to_spec(audio_path, output_path):
    try:
        y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
        if len(y) == 0: return False
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512,
            n_mels=128, fmin=50, fmax=4000)
        db = librosa.power_to_db(mel, ref=np.max)
        mn, mx = db.min(), db.max()
        norm = ((db - mn) / (mx - mn) * 255).astype(np.uint8) if mx > mn else np.zeros_like(db, dtype=np.uint8)
        Image.fromarray(np.flip(norm, axis=0), mode="L").resize(
            (SPEC_W, SPEC_H), Image.Resampling.LANCZOS).convert("RGB").save(str(output_path))
        return True
    except: return False

print("Generating spectrograms...")
spec_records = []
for i, r in enumerate(records):
    sp = SPEC_DIR / f"{Path(r['audio_path']).stem}_spec.png"
    if sp.exists() or audio_to_spec(r['audio_path'], sp):
        r["spectrogram_path"] = str(sp)
        spec_records.append(r)
    if (i+1) % 200 == 0: print(f"  {i+1}/{len(records)}")
records = spec_records
print(f"Generated {len(records)} spectrograms")

In [ ]:
# Cell 6: Instruction pairs + patient-level split
def describe(r):
    if r["label"] == "normal": return "Normal breath sounds. No adventitious sounds."
    parts = []
    if r["has_wheeze"]: parts.append("Wheezing — continuous bright bands 200-1000 Hz")
    if r["has_crackle"]: parts.append("Crackles — discontinuous vertical bright spots")
    return ". ".join(parts) + "."

INSTRUCTION = ("This is a mel-spectrogram of a child's breathing audio.\n"
    "Vertical: frequency (50-4000 Hz). Horizontal: time. Brightness: intensity.\n"
    "Classify the breath sounds as JSON: "
    '{"wheeze": bool, "stridor": bool, "grunting": bool, "crackles": bool, '
    '"normal": bool, "confidence": float, "description": "..."}')

pairs = [{"spectrogram_path": r["spectrogram_path"], "label": r["label"],
    "instruction": INSTRUCTION,
    "response": json.dumps({"wheeze": r["has_wheeze"], "stridor": False, "grunting": False,
        "crackles": r["has_crackle"], "normal": r["label"]=="normal",
        "confidence": 0.9, "description": describe(r)})} for r in records]

pids = list(set(r["patient_id"] for r in records))
random.seed(42); random.shuffle(pids)
train_pats = set(pids[:int(len(pids)*0.8)])
train_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] in train_pats]
test_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] not in train_pats]
print(f"Train: {len(train_pairs)} | Test: {len(test_pairs)}")
for name, sp in [("Train", train_pairs), ("Test", test_pairs)]:
    print(f"  {name}: {dict(Counter(p['label'] for p in sp))}")

In [ ]:
# Cell 7: Load model + processor
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-E4B-it"
print(f"Loading {MODEL_NAME}...")
t0 = time.monotonic()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"),
    torch_dtype=torch.float16,
)
print(f"Loaded in {time.monotonic()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 8: Add LoRA adapter (v2 STRONGER config)
from peft import LoraConfig, get_peft_model, TaskType

# Gradient checkpointing for VRAM
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# Unwrap Gemma4ClippableLinear -> Linear4bit
unwrapped = 0
for name, module in list(model.named_modules()):
    if module.__class__.__name__ == "Gemma4ClippableLinear":
        parts = name.split(".")
        parent = model
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], module.linear)
        unwrapped += 1
print(f"Unwrapped {unwrapped} layers")

# v2: r=32, 4 target modules, dropout for generalization
model = get_peft_model(model, LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()
print(f"VRAM after LoRA: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 9: Build dataset
from torch.utils.data import Dataset as TorchDataset

SYSTEM_MSG = ("You are Malaika, a medical spectrogram analysis assistant following WHO IMCI. "
    "Classify breath sounds from spectrograms. Respond ONLY with JSON. Do NOT use thinking mode.")

class SpectrogramDataset(TorchDataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": spec_img},
                {"type": "text", "text": f"{SYSTEM_MSG}\n\n{pair['instruction']}"},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": pair["response"]},
            ]},
        ]
        inputs = processor.apply_chat_template(
            messages, tokenize=True, return_dict=True,
            return_tensors="pt", add_generation_prompt=False,
        )
        input_ids = inputs["input_ids"].squeeze(0)
        labels = input_ids.clone()
        resp_len = len(tokenizer.encode(pair["response"], add_special_tokens=False))
        if resp_len > 0 and len(labels) > resp_len:
            labels[:-resp_len] = -100
        result = {"labels": labels}
        for key in inputs:
            if inputs[key] is not None:
                result[key] = inputs[key].squeeze(0)
        return result

train_dataset = SpectrogramDataset(train_pairs)
sample = train_dataset[0]
print(f"Dataset: {len(train_dataset)} examples")
print("Fields:")
for k, v in sample.items():
    if hasattr(v, 'shape'): print(f"  {k}: {v.shape} {v.dtype}")

In [ ]:
# Cell 10: Train (v2: 300 steps)
from transformers import TrainingArguments, Trainer

def gemma4_collator(features):
    batch = {}
    for key in features[0].keys():
        values = [f[key] for f in features if key in f and f[key] is not None]
        if not values: continue
        if isinstance(values[0], torch.Tensor):
            try: batch[key] = torch.stack(values)
            except RuntimeError: batch[key] = values[0].unsqueeze(0)
        else: batch[key] = values
    return batch

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./breath_sound_lora_v2",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=300,
        learning_rate=2e-4,
        warmup_steps=20,
        weight_decay=0.01,
        fp16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        seed=42,
        report_to="none",
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    ),
    train_dataset=train_dataset,
    data_collator=gemma4_collator,
)

print(f"Starting v2 training (300 steps, r=32, q/k/v/o_proj)...")
print(f"Estimated time: ~45 min on T4")
t0 = time.monotonic()
result = trainer.train()
train_time = time.monotonic() - t0
print(f"\nDone in {train_time/60:.1f} min | Loss: {result.training_loss:.4f} | Steps: {result.global_step}")

In [ ]:
# Cell 11: Save adapter
ADAPTER_NAME = "malaika-breath-sounds-lora-v2"
model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)
print(f"Saved to {ADAPTER_NAME}/")
for f in sorted(Path(ADAPTER_NAME).glob("*")):
    if f.is_file(): print(f"  {f.name}: {f.stat().st_size/1024/1024:.1f} MB")

In [ ]:
# Cell 12: Evaluate (MUST disable gradient checkpointing first)
model.gradient_checkpointing_disable()
model.eval()

print("=" * 60)
print("EVALUATION ON TEST SET")
print("=" * 60)

correct, total_test = 0, 0
per_label = {"normal": [0,0], "wheeze": [0,0], "crackle": [0,0], "both": [0,0]}

for pair in test_pairs[:30]:
    spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": spec_img},
        {"type": "text", "text": pair["instruction"]}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    )
    # Move ALL tensors to GPU
    inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    pred_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    label = pair["label"]
    per_label[label][1] += 1  # total
    
    try:
        m = re.search(r'\{[^{}]*\}', pred_text)
        if m:
            pred = json.loads(m.group(0))
            exp = json.loads(pair["response"])
            w_ok = pred.get("wheeze") == exp.get("wheeze")
            c_ok = pred.get("crackles") == exp.get("crackles")
            if w_ok and c_ok:
                correct += 1; per_label[label][0] += 1; status = "PASS"
            else:
                status = f"FAIL (w={pred.get('wheeze')} c={pred.get('crackles')})"
        else: status = f"FAIL (no JSON): {pred_text[:60]}"
    except: status = "FAIL (parse)"
    total_test += 1
    print(f"  {status:45s} [{label}]")

print(f"\nOverall: {correct}/{total_test} ({100*correct/total_test:.0f}%)")
print(f"Baseline: 25% (all normal)")
print(f"\nPer-label accuracy:")
for label, (c, t) in per_label.items():
    if t > 0: print(f"  {label:>8s}: {c}/{t} ({100*c/t:.0f}%)")

In [ ]:
# Cell 13: Summary for writeup
print("=" * 60)
print("FINE-TUNING RESULTS (for competition writeup)")
print("=" * 60)
print(f"""\n## Breath Sound Classification via Spectrogram Vision Fine-Tuning

### Innovation
Gemma 4 cannot process audio natively. We convert breath sounds to
mel-spectrogram images (50-4000 Hz, tuned for pediatric respiratory sounds)
and fine-tune Gemma 4's vision capability to classify them.

### Dataset
- ICBHI 2017 Respiratory Sound Database: {len(records)} recordings
- Labels: normal, wheeze, crackle, both (wheeze+crackle)
- Patient-level train/test split (no data leakage)
- Train: {len(train_pairs)} | Test: {len(test_pairs)}

### Method
- Base model: Gemma 4 E4B-it (4-bit quantized)
- Fine-tuning: PEFT LoRA r=32, targets q/k/v/o_proj
- Spectrogram params: sr=22050, n_mels=128, fmin=50, fmax=4000
- Training: 300 steps, lr=2e-4, warmup=20, dropout=0.05
- Hardware: {torch.cuda.get_device_name(0)}
- Training time: {train_time/60:.1f} minutes

### Results
- Training loss: 1.13 -> {result.training_loss:.4f}
- Adapter size: {ADAPTER_NAME}/
- Test accuracy: {correct}/{total_test} ({100*correct/total_test:.0f}%)
- Baseline (no fine-tuning): 25% (model predicts all normal)
""")

print("Per-label:")
for label, (c, t) in per_label.items():
    if t > 0: print(f"  {label}: {c}/{t} ({100*c/t:.0f}%)")

print(f"\nv1 -> v2 comparison:")
print(f"  v1: r=8, q+v only, 100 steps -> 20% (worse than baseline)")
print(f"  v2: r=32, q+k+v+o, 300 steps -> {100*correct/total_test:.0f}%")